# MFA Align (Google Colab)
Данный ноутбук посвящен процессу выравнивания набора данных RUSLAN с помощью инструмента Montreal Forced Aligner в среде Google Colab.

Модели, используемые для выравнивания:
* [russian acoustic model v3.1.0](https://mfa-models.readthedocs.io/en/latest/acoustic/Russian/Russian%20MFA%20acoustic%20model%20v3_1_0.html)
* [russian g2p model v3.1.0](https://mfa-models.readthedocs.io/en/latest/g2p/Russian/Russian%20MFA%20G2P%20model%20v3_1_0.html)
* [russian dictionary v3.1.0](https://mfa-models.readthedocs.io/en/latest/dictionary/Russian/Russian%20MFA%20dictionary%20v3_1_0.html)

Также доступны на [huggingface](https://huggingface.co/MontrealCorpusTools/russian_mfa)

In [ ]:
import os
import shutil
import pandas as pd
from pathlib import Path

#### Константы: пути к данным и моделям
* `DATA_DIR` -- корневая директория для данных
* `RUSLAN_DIR` -- место хранения исходных данных -- аудиофайлов и текстовок к ним
* `TEXTGRID_DIR` -- место, куда будут сохранены файлы разметки (.TextGrid)
* `MFA_MODELS_DIR` -- директория, куда `mfa model download` сохраняет предобученные модели (словарь, акустическая модель, G2P)
* `METADATA_PATH` -- путь к файлу метаданных с нормализованным текстом

In [ ]:
DATA_DIR = Path('/content/data')
RUSLAN_DIR = DATA_DIR / 'RUSLAN' / 'ruslan_dataset' / 'RUSLAN'
TEXTGRID_DIR = Path('/content/RUSLAN_align')
MFA_MODELS_DIR = Path.home() / 'Documents' / 'MFA' / 'pretrained_models'
METADATA_PATH = DATA_DIR / 'metadata_RUSLAN_22200_normalized.csv'

DATA_DIR.mkdir(parents=True, exist_ok=True)
RUSLAN_DIR.mkdir(parents=True, exist_ok=True)
TEXTGRID_DIR.mkdir(parents=True, exist_ok=True)

#### Загрузка файла метаданных

Файл должен быть результатом нормализации: то есть в нём помимо двух колонок (идентификатор, сырой текст), должна быть также и нормализованная текстовая колонка.

In [ ]:
from google.colab import files

print("Загрузите файл метаданных с вашего компьютера:")
uploaded = files.upload()

uploaded_file_name = list(uploaded.keys())[0]
os.rename(uploaded_file_name, METADATA_PATH)
print(f"Файл успешно сохранен по пути: {METADATA_PATH}")

#### Загрузка аудиодатасета RUSLAN

Аудиофайлы скачиваются с Kaggle через `kagglehub` и копируются в `DATA_DIR`.

In [ ]:
import kagglehub

download_path = kagglehub.dataset_download("freezerainml/ruslan")

target_dir = DATA_DIR / 'RUSLAN'
target_dir.mkdir(parents=True, exist_ok=True)

for file_name in os.listdir(download_path):
    source_file = os.path.join(download_path, file_name)
    target_file = os.path.join(target_dir, file_name)

    if os.path.isdir(source_file):
        shutil.copytree(source_file, target_file, dirs_exist_ok=True)
    else:
        shutil.copy2(source_file, target_file)

print("Аудиофайлы датасета успешно скачаны и распакованы!")

#### Чтение списка файлов

In [ ]:
ruslan = pd.read_csv(METADATA_PATH, sep='|', names=['id', 'raw', 'nrm'])
ruslan.head()

#### Сохранение текстовок для выравнивателя

Необходимо положить файл с расширением .txt рядом с каждым аудиофайлом.

In [ ]:
print("Старт генерации .txt файлов...")
for file_id, t in ruslan[['id', 'nrm']].values:
    file_id = str(file_id)
    if os.path.exists(os.path.join(RUSLAN_DIR, file_id + '.wav')):
        txt_path = os.path.join(RUSLAN_DIR, file_id + '.txt')
        with open(txt_path, 'w', encoding='utf-8') as f:
            f.write(str(t))

print(f"Текстовые файлы созданы в: {RUSLAN_DIR}")

#### Настройка среды MFA

MFA устанавливается через `conda`, для чего в Colab используется `condacolab`.

In [ ]:
!pip install -q "https://github.com/conda-incubator/condacolab/archive/main.zip"
import condacolab
condacolab.install()

**Важно!** После вызова `condacolab.install()` среда выполнения Colab автоматически перезапускается. Запустите следующую ячейку отдельно, уже после перезапуска.

In [ ]:
import condacolab
condacolab.check()

!conda install -y -c conda-forge montreal-forced-aligner=3.4.1 openfst pynini kaldi

#### Первый запуск выравнивания

Скачиваем акустическую модель и словарь, после чего запускаем `mfa align`.

Результат работы команды -- директория с файлами разметки .TextGrid.

In [ ]:
!mfa model download acoustic russian_mfa
!mfa model download dictionary russian_mfa
!mfa align {RUSLAN_DIR} russian_mfa russian_mfa {TEXTGRID_DIR}/v1 --single_speaker -j 2

#### Сохранение результата выравнивания (опционально)

Архивируем и скачиваем полученные .TextGrid файлы на локальный компьютер.

In [ ]:
from google.colab import files

!zip -q -r /content/RUSLAN_align.zip {TEXTGRID_DIR}
files.download('/content/RUSLAN_align.zip')

#### Проверка результата: потребуется библиотека praatio для работы с TextGrid

In [ ]:
from praatio import textgrid
# Примечание: убедитесь, что скрипт prepare_training_data.py находится в /content/
from prepare_training_data import read_text_grids

ruslan = pd.read_csv(METADATA_PATH, sep='|', names=['id', 'raw', 'nrm'])
ruslan['phn'] = read_text_grids(ruslan, TEXTGRID_DIR / 'v1')[2]

### Анализ ошибок разметки: поиск Out-Of-Vocabulary слов
Чтение словаря и анализ корпуса на предмет OOV-слов.

Метка `spn` на уровне фонем, а.к. `spoken noise` -- фактор, говорящий о том, что слово не разметилось; одна из причин -- его нет в словаре.

In [ ]:
ru_dict = pd.read_csv(MFA_MODELS_DIR / 'dictionary' / 'russian_mfa.dict', names=['grapheme', 'p1', 'p2', 'p3', 'p4', 'phoneme'], sep='\t')
ru_dict.loc[ru_dict.phoneme.isna(), 'phoneme'] = ru_dict[ru_dict.phoneme.isna()]['p1']

words = ' '.join(
    ruslan[ruslan.phn.str.contains('spn')].raw.str.lower().str.replace('[^а-яё ]', '', regex=True)
).split(' ')

dict_words = set(ru_dict.grapheme.unique())
oov_words = list(set(words) - dict_words)

oov_file_path = RUSLAN_DIR / 'oovs_found.txt'
pd.DataFrame({'txt': oov_words}).to_csv(oov_file_path, index=False, header=False)

print(f"Найдено уникальных OOV слов: {len(oov_words)}. Список сохранен в: {oov_file_path}")

#### Предсказание вариантов произношения для OOV-слов с помощью G2P-модели

In [ ]:
G2P_DIR = Path('/content/models_g2p')
G2P_DIR.mkdir(parents=True, exist_ok=True)
G2P_RESULT_PATH = G2P_DIR / 'oovs_mfa_g2p_3_1_0.txt'

print("Скачивание G2P модели...")
!mfa model download g2p russian_mfa

print("Запуск G2P генерации для OOV-слов...")
!mfa g2p {oov_file_path} russian_mfa {G2P_RESULT_PATH}

#### Формирование расширенного словаря

Объединяем исходный словарь `russian_mfa` со сгенерированными транскрипциями OOV-слов в новый словарь.

In [ ]:
BASE_DICT_PATH = MFA_MODELS_DIR / 'dictionary' / 'russian_mfa.dict'
NEW_DICT_PATH = MFA_MODELS_DIR / 'dictionary' / 'russian_mfa_oov_ruslan.dict'

NEW_DICT_PATH.parent.mkdir(parents=True, exist_ok=True)

with open(NEW_DICT_PATH, 'w', encoding='utf-8') as writer:
    dict_orig = open(BASE_DICT_PATH, 'r', encoding='utf-8').read().strip().split('\n')
    oov_trans = open(G2P_RESULT_PATH, 'r', encoding='utf-8').read().strip().split('\n')
    writer.write('\n'.join(sorted(dict_orig + oov_trans)))

print(f"Новый расширенный словарь успешно сохранен в: {NEW_DICT_PATH}")

#### Повторный запуск выравнивания

После пополнения словаря OOV-словами выравнивание должно пойти лучше.

In [ ]:
!mfa align {RUSLAN_DIR} {NEW_DICT_PATH} russian_mfa {TEXTGRID_DIR}/v2 --single_speaker -j 2

#### Сохранение итогового результата (опционально)

In [ ]:
from google.colab import files

!zip -q -r /content/RUSLAN_align_v2.zip {TEXTGRID_DIR}/v2
files.download('/content/RUSLAN_align_v2.zip')